In [102]:
import pandas as pd
import json
import importlib
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")
import numpy as np

# load the data

In [117]:
concept_root = "../data/concept/"
out_concept_root = "../data/outside_concept/"
response_root = "../data/respondent/"
output_root = "../data/picking_result/"

In [3]:
# take the concepts 
with open(concept_root + 'new_cid_concept_us_food.json', 'r') as f:
    food_concepts = json.load(f)

# all concepts 
all_us_food_concepts = pd.read_excel(out_concept_root + '0407_cleaned_us_food_concepts.xlsx')

# open transformed
with open(response_root + 'transformed_0407_id_normal_interview.json', 'r', encoding='utf-8') as f:
    transformed_respondent = json.load(f)

response_table = pd.read_excel(response_root + "response_table_us_food.xlsx")
response_table.drop(columns=['id'], inplace=True)

In [4]:
import sys
sys.path.append("../")
from models import similar as sm
from models import need_filter as nf

c:\Users\Yuding.Duan\OneDrive - Ipsos\3. self_projects\llm_synthetic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [124]:
import importlib
importlib.reload(nf)
importlib.reload(sm)

<module 'models.similar' from 'c:\\Users\\Yuding.Duan\\OneDrive - Ipsos\\3. self_projects\\llm_synthetic\\combination\\..\\models\\similar.py'>

# go for the whole process

**prepare all the data**

- food_concepts  
- all_us_food_concepts  
- transformed_respondent

In [6]:
us_food_cates = list(set(all_us_food_concepts.dropna(subset=['CAT2'])['CAT2']))
# Example usage with caching:
corpus = us_food_cates[:]
# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"


In [7]:
all_us_food_concepts.drop_duplicates(subset=['ConceptText'], inplace=True)
all_us_food_concepts.reset_index(drop=True, inplace=True)

In [ ]:
kpi_mapping = {
    "relevance": "RelFlag",
    "differentiation": "DiffFlag",
    "believability": "BelFlag"
}

In [43]:
# cleaning 
ids_set = set(response_table['ids']) & set(transformed_respondent.keys())
response_table = response_table[response_table['ids'].isin(ids_set)]; response_table.reset_index(drop=True, inplace=True)

let's go

**ob_type**: `real` or `synthetic_contra` or `synthetic_same`

In [51]:
# parameters 
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl" # vector cache path
cate_match_bound = 0.5 # this is for category matching
searcher_cate = sm.SimilaritySearcher()
searcher_cate.fit(us_food_cates, cache_path=CACHE_PATH)


## boundary stuff 
contra_lower_bound = 0.28
same_lower_bound = 0.5
same_upper_bound = 0.75

In [ ]:
# final_result = {'ids':[], 'concept':[], 'question':[], 'answer':[], 'ob_type':[], 'corresponding_concept':[]}
# final_results = [] 

In [89]:
id_kpi_pairs = set(list(zip(response_table['ids'], response_table['question'])))
# id_kpi_pairs = list(id_kpi_pairs)[:100]

In [ ]:
final_result = {'ids':[], 'concept':[], 'question':[], 'supposed_answer':[], 'ob_type':[], 'corresponding_concept':[]}
for id, kpi in tqdm(id_kpi_pairs, total=len(id_kpi_pairs)):
    mask = (response_table['question'] == kpi) &(response_table['ids']==id)
    sub = response_table[mask]
    # final_results.append(sub)

    if len(sub['answer'].value_counts()) == 1:
        process_type = 'contra' # to get the opposite answer
    else:
        process_type = 'same' # to get the same answer


    for i in range(len(sub)):
        item = sub.iloc[i]

        # to get the category 
        if process_type == 'contra':
            query = food_concepts[str(item['concept'])]['concept_Cate']
            top_results, bottom_results = searcher_cate.search(query, top_n=5, bottom_m=0)
            suitable_cates = set([k[0] for k in top_results if k[1] >= cate_match_bound])

            if item['answer'] == 'yes':
                out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
            else:
                out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['H', 'MH']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
            
            filted_concepts = all_us_food_concepts[out_mask]
            corpus = list(filted_concepts['ConceptText'])


            searcher_concept = sm.SimilaritySearcher()
            searcher_concept.fit(corpus, cache_path=CACHE_PATH)

            query = food_concepts[str(item['concept'])]['concept_content']
            top_results, bottom_results = searcher_concept.search(query, top_n=3, bottom_m=3)
            candidate = [item[0] for item  in bottom_results if item[1]<= contra_lower_bound]
            if not candidate:
                continue
            if item['answer'] == "yes":
                pick_answer = 'no'    
            else:
                pick_answer = 'yes'
            
            for k in range(len(candidate)):
                final_result['ids'].append(id)
                final_result['concept'].append(candidate[k])
                final_result['question'].append(kpi)
                final_result['supposed_answer'].append(pick_answer)
                final_result['ob_type'].append('synthetic_contra')
                final_result['corresponding_concept'].append(str(item['concept']))
        else: 

            if item['answer'] == 'yes':
                out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['H', 'MH'])
            else:
                out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) 
            
            filted_concepts = all_us_food_concepts[out_mask]
            corpus = list(filted_concepts['ConceptText'])


            searcher_concept = sm.SimilaritySearcher()
            searcher_concept.fit(corpus, cache_path=CACHE_PATH)

            query = food_concepts[str(item['concept'])]['concept_content']
            top_results, bottom_results = searcher_concept.search(query, top_n=3, bottom_m=3)
            candidate = [item[0] for item  in top_results if same_lower_bound<= item[1]<= same_upper_bound]
            if not candidate:
                continue
            
            pick_answer = item['answer']
            for k in range(len(candidate)):
                final_result['ids'].append(id)
                final_result['concept'].append(candidate[k])
                final_result['question'].append(kpi)
                final_result['supposed_answer'].append(pick_answer)
                final_result['ob_type'].append('synthetic_same')
                final_result['corresponding_concept'].append(str(item['concept']))


100%|██████████| 2340/2340 [03:55<00:00,  9.96it/s]


In [107]:
final_result = pd.DataFrame(final_result)

In [109]:
final_result.head(5)

,ids,concept,question,supposed_answer,ob_type,corresponding_concept
0,3de63060-c9eb-11ef-8154-dfb6be1747bc,Chester Cheetle Painting includes a Chester-sh...,believability,no,synthetic_contra,9
1,3de63060-c9eb-11ef-8154-dfb6be1747bc,Now the nourishing power of chicken broth can ...,believability,no,synthetic_contra,9
2,3de63060-c9eb-11ef-8154-dfb6be1747bc,Entenmann's and Dr. Pepper are teaming up for ...,believability,no,synthetic_contra,9
3,3de63060-c9eb-11ef-8154-dfb6be1747bc,A line of frozen quesadillas that are ready wh...,believability,no,synthetic_contra,11
4,3de63060-c9eb-11ef-8154-dfb6be1747bc,Swanson Finishing Sauce is a smooth savory sau...,believability,no,synthetic_contra,11


In [ ]:
transformed_respondent

In [ ]:
# spliting and run 
split_num = 300
final_result_dfs = np.array_split(final_result, split_num)
for i in range(26, split_num):
    print(f"Processing split {i+1}/{split_num}...")
    df = final_result_dfs[i]
    items = []
    for idx, row in df.iterrows():
        respondent_info = transformed_respondent[row['ids']]
        items.append(
            {"new_concept": row['concept'], 'kpi_type':row['question'], 'system_info':respondent_info, "return_reasoning": False}
        )
    batch_results = await nf.ai_filter_batch_async(
        items,
        max_concurrency=8,
        show_progress=True,
        progress_desc="AI filter"
    )
    df = df.loc[df['supposed_answer']==batch_results]
    df.rename(columns={'supposed_answer':'answer'}, inplace=True)
    df.to_excel(output_root + f"picking_result_{i+1}.xlsx", index=False)
    



# get the final trainning set 

In [127]:
del df 

In [128]:
dfs = []
for i in range(1, 301):
    df = pd.read_excel(output_root + f"picking_result_{i}.xlsx")
    dfs.append(df)


In [129]:
dfs = pd.concat(dfs, ignore_index=True)

In [132]:
response_table['ob_type'] = 'real'
response_table['corresponding_concept'] = response_table['concept']

In [133]:
final_set = pd.concat([response_table, dfs], ignore_index=True)

In [136]:
final_set['ob_type'].value_counts()

ob_type
synthetic_contra    5320
real                4506
synthetic_same       974
Name: count, dtype: int64

In [141]:
final_set.to_excel('../data/final_synthetic_and_real_dataset.xlsx', index=False)

In [139]:
kpi = 'relevance'
final_set[['question', 'answer', 'ob_type', 'ids']].groupby(['question', 'ob_type', 'answer']).count()

ids
question        ob_type          answer      
believability   real             no       148
                                 yes     1354
                synthetic_contra no      1151
                                 yes      176
                synthetic_same   no       109
                                 yes       48
differentiation real             no       425
                                 yes     1077
                synthetic_contra no      2303
                                 yes       58
                synthetic_same   no       381
                                 yes       16
relevance       real             no       717
                                 yes      785
                synthetic_contra no      1183
                                 yes      449
                synthetic_same   no       322
                                 yes       98

In [144]:
final_set.shape

(10800, 6)

In [155]:
final_set.head()

,ids,concept,question,answer,ob_type,corresponding_concept
0,101daf40-ed83-11ee-905c-7d576dd0d5d9,4,relevance,yes,real,4
1,101daf40-ed83-11ee-905c-7d576dd0d5d9,12,relevance,yes,real,12
2,101daf40-ed83-11ee-905c-7d576dd0d5d9,4,differentiation,yes,real,4
3,101daf40-ed83-11ee-905c-7d576dd0d5d9,12,differentiation,yes,real,12
4,101daf40-ed83-11ee-905c-7d576dd0d5d9,4,believability,yes,real,4


In [157]:
id = '101daf40-ed83-11ee-905c-7d576dd0d5d9'
kpi = 'relevance'

mask = (final_set['ids'] == id) & (final_set['question'] == kpi)
final_set[mask]

,ids,concept,question,answer,ob_type,corresponding_concept
0,101daf40-ed83-11ee-905c-7d576dd0d5d9,4,relevance,yes,real,4
1,101daf40-ed83-11ee-905c-7d576dd0d5d9,12,relevance,yes,real,12
5932,101daf40-ed83-11ee-905c-7d576dd0d5d9,Boom Chicka Pop Kettle Corn – 6 count (6) Boom...,relevance,no,synthetic_contra,4
5933,101daf40-ed83-11ee-905c-7d576dd0d5d9,NEW Garden Veggie Pasta-Flavored Puffs are tri...,relevance,no,synthetic_contra,4
5934,101daf40-ed83-11ee-905c-7d576dd0d5d9,NEW Garden Veggie League of Champions are the ...,relevance,no,synthetic_contra,4
5935,101daf40-ed83-11ee-905c-7d576dd0d5d9,Jimmy Dean Sweet Bacon Jam A sweet and smokey ...,relevance,no,synthetic_contra,12
5936,101daf40-ed83-11ee-905c-7d576dd0d5d9,"Jimmy Dean Breakfast Sausage Marinara Rich, re...",relevance,no,synthetic_contra,12


In [159]:
final_result.iloc[[0]]

,ids,concept,question,supposed_answer,ob_type,corresponding_concept
0,3de63060-c9eb-11ef-8154-dfb6be1747bc,Chester Cheetle Painting includes a Chester-sh...,believability,no,synthetic_contra,9


In [163]:
sub_results = []

for id in set(final_set['ids']):
    for kpi in set(final_set['question']):
        mask = (final_set['ids'] == id) & (final_set['question'] == kpi)
        sub = final_set[mask]

        sub_real = sub[sub['ob_type']=='real']
        sub_results.append(sub_real)
        sub_synthetic = sub[sub['ob_type']!='real']
        for concept in sub_real['concept']:
            try:
                sub_results.append(sub_synthetic.loc[sub_synthetic['corresponding_concept'] == concept].iloc[[0]])
            except:
                pass


In [169]:
fnl_result = pd.concat(sub_results)

In [173]:
fnl_result[['question', 'answer', 'ids']].groupby(['question', 'answer']).count()

ids
question        answer      
believability   no      1018
                yes     1456
differentiation no      1456
                yes     1131
relevance       no      1409
                yes     1102

In [177]:
11/26

0.4230769230769231